# Did the geometry fixes change any published result?

Two bugs were found in the board and throw geometry:

1. **The coordinate grids used `np.linspace`.** `np.linspace(-r, r, n)` has spacing
   $2r/(n-1)$, not the $2r/n$ that `mm_per_pixel` assumes, and for even $n$ it straddles
   zero instead of landing on a pixel. So the board was drawn ~0.05% too large with its
   bull half a pixel off-centre, and the throwing distribution was centred up to a pixel
   away from the requested aim point, with the error growing towards the edge of the board.
2. **A latent gap in the segment definitions.** The angular intervals are half-open and
   `arctan2` returns exactly $+\pi$ on the negative x-axis, so a pixel landing exactly
   there would match no segment and score 0.

The second never fired in the published work -- on an even-length `linspace` grid $y = 0$
is never sampled -- but it is closed anyway.

The first *did* change numbers. This notebook quantifies by how much, by reconstructing
the pre-fix code and running both versions side by side.

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

import darts.stats as stats
from darts.dartboards import DARTBOARD_CONSTANTS as C, QUADRO_CONSTANTS, generate_dartboard
from darts.utils import region_label, mm_per_pixel

PIXELS = 2048                       # the resolution the blog post used
MMPP = mm_per_pixel(PIXELS)


def old_generate_dartboard(pixels, quadro=False):
    '''generate_dartboard as it was before the fix: linspace grid, unwrapped theta.'''
    radius = C['DARTBOARD_RADIUS_MM']
    x, y = np.meshgrid(np.linspace(-radius, radius, pixels),
                       np.linspace(-radius, radius, pixels))
    r = np.sqrt(x * x + y * y)
    theta = np.arctan2(y, x)
    bi, bo = C['INNER_BULLSEYE_RADIUS_MM'], C['OUTER_BULLSEYE_RADIUS_MM']
    di, do = C['DOUBLE_INNER_RADIUS'], C['DOUBLE_OUTER_RADIUS']
    ti, to = C['TRIPLE_INNER_RADIUS'], C['TRIPLE_OUTER_RADIUS']
    single = (((r >= bo) & (r < ti)) | ((r >= to) & (r < di))).astype(int)
    double = ((r >= di) & (r < do)).astype(int)
    triple = ((r >= ti) & (r < to)).astype(int)
    mult = single + 2 * double + 3 * triple
    arr = np.zeros([pixels, pixels])
    for score, intervals in C['SEGMENTS'].items():
        seg = np.zeros([pixels, pixels], bool)
        for lo, hi in intervals:
            seg |= (theta < hi * np.pi) & (theta >= lo * np.pi)
        arr += score * seg * mult
    arr += 25 * ((r >= bi) & (r < bo)) + 50 * (r < bi)
    return arr, np.logical_or(double, (r < bi))


def old_gaussian_filter(board, mu, Sigma):
    '''gaussian_filter as it was before the fix: linspace coordinate grid.'''
    pixels = board.shape[0]
    xx = np.linspace(-board.shape[0] // 2, board.shape[0] // 2, pixels)
    x = np.empty((xx.size, xx.size), dtype=xx.dtype)
    for j in range(xx.size):
        x[:, j] = xx[j]
    y = x.T
    det = Sigma[0, 0] * Sigma[1, 1] - Sigma[0, 1] ** 2
    expo = (Sigma[1, 1] * (x - mu[1]) ** 2
            - 2 * Sigma[0, 1] * (x - mu[1]) * (y - mu[0])
            + Sigma[0, 0] * (y - mu[0]) ** 2) / det
    return np.exp(-expo / 2.0) / (2 * np.pi * np.sqrt(det))


real_gauss = stats.gaussian_filter
new_board, _ = generate_dartboard(PIXELS)
old_board, _ = old_generate_dartboard(PIXELS)
print(f'{PIXELS}px boards built; {(new_board != old_board).sum():,} pixels differ '
      f'({100 * (new_board != old_board).mean():.3f}%)')

2048px boards built; 13,923 pixels differ (0.332%)


Most of those differing pixels are on region boundaries, reclassified because the board is
drawn 0.05% larger and half a pixel over. First, confirm the latent segment bug really was
latent -- the old board should have no dead pixels *in its own coordinates*.

In [2]:
lin = np.linspace(-C['DARTBOARD_RADIUS_MM'], C['DARTBOARD_RADIUS_MM'], PIXELS)
gx, gy = np.meshgrid(lin, lin)
r_old = np.hypot(gx, gy)
theta_old = np.arctan2(gy, gx)
scoring = (r_old >= C['OUTER_BULLSEYE_RADIUS_MM']) & (r_old < C['DOUBLE_OUTER_RADIUS'])
print(f'pixels with theta exactly == pi on the old grid: {(theta_old == np.pi).sum()}')
print(f'dead (zero-score) pixels in the old board, its own coordinates: '
      f'{(scoring & (old_board == 0)).sum()}')

pixels with theta exactly == pi on the old grid: 0
dead (zero-score) pixels in the old board, its own coordinates: 0


Zero, as expected: the bug could not fire. Now the quantity the post actually reports --
the maximum expected score and where to aim for it.

In [3]:
def analyse(board, gauss, sigma_mm):
    stats.gaussian_filter = gauss
    sp = sigma_mm / MMPP
    Sigma = sp * sp * np.eye(2)
    exp = stats.expected_score(board, np.zeros(2), Sigma, padding=200)
    i, j = np.unravel_index(exp.argmax(), exp.shape)
    std = stats.std_score(board, np.zeros(2), Sigma, padding=200)
    stats.gaussian_filter = real_gauss
    return exp[i, j], (i, j), std[i, j]


rows = []
for s in [3, 5, 6.5, 8, 10, 12, 15, 16.5, 17, 18, 20, 25, 30, 40]:
    eo, po, so = analyse(old_board, old_gaussian_filter, s)
    en, pn, sn = analyse(new_board, real_gauss, s)
    rows.append({'sigma (mm)': s,
                 'old E[score]': round(eo, 4), 'old aim': region_label(po, PIXELS),
                 'new E[score]': round(en, 4), 'new aim': region_label(pn, PIXELS),
                 'change %': round(100 * (en - eo) / eo, 3),
                 'old std': round(so, 3), 'new std': round(sn, 3)})
pd.DataFrame(rows).set_index('sigma (mm)')

,old E[score],old aim,new E[score],new aim,change %,old std,new std
sigma (mm),,,,,,,
3.0,52.8595,T20,52.6624,T20,-0.373,15.318,15.484
5.0,43.1094,T20,42.9257,T20,-0.426,19.781,19.809
6.5,38.1160,T20,37.9925,T20,-0.324,20.170,20.159
8.0,34.0657,T20,33.9778,T20,-0.258,20.011,19.987
10.0,29.5550,T20,29.4964,T20,-0.198,19.491,19.446
12.0,25.9056,T20,25.8636,T20,-0.162,18.733,18.705
15.0,21.7783,T20,21.7513,T20,-0.124,17.431,17.406
16.5,20.2069,T20,20.1846,T20,-0.110,16.767,16.744
17.0,19.7443,T20,19.7474,T19,0.016,16.548,15.067


The expected score moves by well under half a percent, and **the recommended region is
identical at every $\sigma$ except right at the treble 20 / treble 19 crossover**. The
$\sigma = 6.5$ mm peak in the standard deviation survives too.

So the one published number that needs amending is where that crossover sits. The post says
"for players with $\sigma < 17$ mm, the treble 20 is the best place to aim". Let us find it
precisely in both versions.

In [4]:
rows = []
for s in np.arange(16.4, 17.5, 0.1):
    _, po, _ = analyse(old_board, old_gaussian_filter, s)
    _, pn, _ = analyse(new_board, real_gauss, s)
    rows.append({'sigma (mm)': round(s, 2), 'old': region_label(po, PIXELS),
                 'new': region_label(pn, PIXELS)})
cross = pd.DataFrame(rows).set_index('sigma (mm)')
cross

,old,new
sigma (mm),,
16.4,T20,T20
16.5,T20,T20
16.6,T20,T20
16.7,T20,T20
16.8,T20,T19
16.9,T20,T19
17.0,T20,T19
17.1,T19,T19
17.2,T19,T19


The crossover moves from just above 17 mm to just below 16.8 mm. That is the single
correction worth making to the published post: **"$\sigma < 17$ mm" becomes
"$\sigma < 16.8$ mm"**. Everything else stands.

## The stored MDP results

`notebooks/inf-round_1-player/` holds solved 1-player MDPs at coarser resolutions. Note
the folder names give $\sigma$ in **pixels**, not millimetres. Recomputing them with the
old and new geometry on the *same* aiming grid isolates the effect of the fixes.

In [5]:
from darts.transitions import transition_maps


def solve_one_dart(P, CP, S, G=501):
    '''Memoryless 1-dart MDP; the self-loop from a wasted dart is solved in closed form.'''
    V = np.zeros(G + 1); pol = np.zeros(G + 1, dtype=int)
    for u in range(2, G + 1):
        k = int(np.searchsorted(S, u - 2, side='right'))
        w = np.zeros(len(S)); w[1:k] = V[u - S[1:k]]
        co_u = CP[:, S == u].ravel() if (S == u).any() else np.zeros(P.shape[0])
        prog = P[:, 1:k].sum(axis=1) + co_u
        q = np.where(prog > 1e-12, (P @ w - 1.0) / np.maximum(prog, 1e-300), -np.inf)
        V[u] = q.max(); pol[u] = q.argmax()
    return V, pol


rows = []
for name, px, sig_px in [('128pixel_8sigma', 128, 8), ('256pixel_12sigma', 256, 12)]:
    path = f'../inf-round_1-player/{name}'
    stored = pd.read_csv(f'{path}/values.csv').set_index('state')['value']
    pts = pd.read_csv(f'{path}/points.csv', index_col=0).values.astype(int)
    Sigma = float(sig_px) ** 2 * np.eye(2)
    res = {}
    for tag, board_fn in [('old', old_generate_dartboard), ('new', generate_dartboard)]:
        bd, ck = board_fn(px)
        pm, cm, S = transition_maps(bd, ck, Sigma)
        P = np.ascontiguousarray(pm[:, pts[:, 0], pts[:, 1]].T)
        CP = np.ascontiguousarray(cm[:, pts[:, 0], pts[:, 1]].T)
        res[tag] = solve_one_dart(P, CP, S)
    (Vo, po), (Vn, pn) = res['old'], res['new']
    changed = sum(region_label(pts[po[u]], px) != region_label(pts[pn[u]], px)
                  for u in range(2, 502))
    rows.append({'run': name, 'sigma (mm)': round(sig_px * mm_per_pixel(px), 1),
                 'stored CSV': round(-stored[501], 3),
                 'old geometry': round(-Vo[501], 3), 'new geometry': round(-Vn[501], 3),
                 'change %': round(100 * abs(Vn[501] - Vo[501]) / abs(Vo[501]), 2),
                 'mean |delta| darts': round(np.abs(Vn[2:] - Vo[2:]).mean(), 4),
                 'aim region changed at': f'{changed}/500'})
pd.DataFrame(rows).set_index('run')

,sigma (mm),stored CSV,old geometry,new geometry,change %,mean |delta| darts,aim region changed at
run,,,,,,,
128pixel_8sigma,28.2,46.265,47.089,46.870,0.46,0.1646,436/500
256pixel_12sigma,21.1,36.890,36.814,36.714,0.27,0.1094,13/500


The recomputation with the old geometry reproduces the stored CSVs closely, which
validates the reconstruction. The fixes then move the expected-darts figure by 0.3--0.5%.

The interesting split is in the last column. At 256 pixels only a handful of aiming
recommendations move. At 128 pixels almost all of them do -- but the *values* barely change
(a mean of 0.16 darts). That is the near-tie effect: on a board too coarse to resolve an
8 mm bed, many aiming points are worth almost exactly the same, so the argmax flips for
reasons worth nothing. It is a warning about reading too much into a recommendation without
also looking at its margin.

## Conclusion

* **No published conclusion changes.** Expected scores move by <0.5%, recommended regions
  are unchanged except at the crossover, and the $\sigma = 6.5$ mm standard-deviation peak
  survives.
* **One number to amend**: the treble 20 / treble 19 crossover, 17 mm -> 16.8 mm.
* The latent segment bug never affected anything, because the old grid never sampled the
  axis where it would have fired.